<a href="https://colab.research.google.com/github/Coder-Pinku/Data_Fundamental/blob/main/Info_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import os
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Custom request headers to avoid being blocked by basic bot protections
REQUEST_HEADERS = {
    "User-Agent": "MyWebScraper/1.0"
}


def read_url_list(file_path: str) -> list:
    """
    Read a text file line by line and collect all valid URLs.
    Only lines starting with 'http' are treated as URLs.
    """
    urls = []

    # If file does not exist, just return an empty list
    if not os.path.isfile(file_path):
        return urls

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line.startswith("http"):
                urls.append(line)

    return urls


def fetch_soup(url: str) -> BeautifulSoup | None:
    """
    Send a GET request to the given URL and return a BeautifulSoup object.
    Returns None if the request fails for any reason.。
    """
    try:
        response = requests.get(url, headers=REQUEST_HEADERS, timeout=15)
        response.raise_for_status()  # Raise error for HTTP codes like 4xx/5xx
        return BeautifulSoup(response.text, "html.parser")
    except Exception:
        # In a real project you might log the error instead of silent pass
        return None


def extract_country_and_rank(soup: BeautifulSoup) -> pd.DataFrame:
    """
    From the main countries listing page, extract country names and overall rank.
    """
    country_cards = soup.select("div.picTrans.recordsetContainer")

    countries = []
    ranks = []

    for card in country_cards:
        try:
            # Country name
            name_el = card.find("span", class_="textWhite textLarge textShadow")
            # Global rank value
            rank_el = card.find("span", class_="textWhite textLarge textBold")

            country_name = name_el.text.strip()
            rank_value = rank_el.text.strip()

            countries.append(country_name)
            ranks.append(rank_value)
        except Exception:
            # Skip blocks that don't match the expected structure
            continue

    return pd.DataFrame(
        {
            "Country": countries,
            "Rank": ranks,
        }
    )


def extract_metric_from_page(url: str) -> pd.DataFrame | None:
    """
    For a specific Global Firepower metric page:
    - Parse all country entries
    - Extract the metric value for each country
    - Return a DataFrame with 'Country' and one metric column
    """
    soup = fetch_soup(url)
    if soup is None:
        return None

    metric_cards = soup.select("div.picTrans.recordsetContainer")
    if not metric_cards:
        return None

    countries = []
    values = []

    for card in metric_cards:
        try:
            # Country name
            name_el = card.find("span", class_="textWhite textLarge textShadow")
            # Metric value is usually in the last 'textWhite textLarge' span
            value_el = card.find_all("span", class_="textWhite textLarge")[-1]

            country_name = name_el.text.strip()
            metric_value = value_el.text.strip()

            countries.append(country_name)
            values.append(metric_value)
        except Exception:
            # Ignore entries that do not have the expected structure
            continue

    # Generate a usable metric column name from the URL
    metric_name = (
        url.split("/")[-1]  # get last part of URL
        .replace(".php", "")
        .replace("-", "_")
    )

    return pd.DataFrame(
        {
            "Country": countries,
            metric_name: values,
        }
    )


def clean_numeric_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    For all columns except 'Country' and 'Rank', try to:
    - Remove commas
    - Extract the numeric part
    Resulting values are left as strings (you can cast to float if needed).
    """
    for col in df.columns[2:]:
        df[col] = (
            df[col]
            .astype(str)  # Ensure string for regex operations
            .str.replace(",", "", regex=False)  # Remove thousands separators
            .str.extract(r"(\d+(?:\.\d+)?)")[0]  # Capture numeric pattern
        )
    return df


def build_global_firepower_dataset() -> pd.DataFrame:
    """
    Main function to:
    - Load list of metric URLs from file
    - Scrape base ranking page
    - Scrape each metric page and merge into a single DataFrame
    - Clean numeric columns
    """
    base_url = "https://www.globalfirepower.com/countries-listing.php"
    metric_urls = read_url_list("links_for_military_data.txt")
    print("Metric URLs found:", len(metric_urls), metric_urls) #debug

    # 1. Get base page data (country + rank)
    base_soup = fetch_soup(base_url)
    if base_soup is None:
        raise RuntimeError("Failed to load base countries listing page.")

    master_df = extract_country_and_rank(base_soup)
    print("Base columns:", master_df.columns.tolist())

    # 2. Loop through each metric page and merge its data
    for url in metric_urls:
      print("Scraping:",url)
      metric_df = extract_metric_from_page(url)
      if metric_df is None or metric_df.empty:
        print(" -> no data extracted from this page")
        continue
      print(" -> columns extracted:", metric_df.columns.tolist())

      # Merge on 'Country' so that each metric becomes a new column
      master_df = master_df.merge(metric_df, on="Country", how="left")

    # 3. Clean numeric-looking columns
    master_df = clean_numeric_columns(master_df)
    print("final columns:", master_df.columns.tolist())

    return master_df


if __name__ == "__main__":
    # Build the final dataset by scraping all pages
    final_df = build_global_firepower_dataset()

    # Save raw scraped data to CSV
    output_file = "military_raw_data.csv"
    final_df.to_csv(output_file, index=False)

    print(f"{output_file} created successfully!")

Metric URLs found: 54 ['https://www.globalfirepower.com/total-population-by-country.php', 'https://www.globalfirepower.com/available-military-manpower.php', 'https://www.globalfirepower.com/manpower-fit-for-military-service.php', 'https://www.globalfirepower.com/manpower-reaching-military-age-annually.php', 'https://www.globalfirepower.com/active-military-manpower.php', 'https://www.globalfirepower.com/active-reserve-military-manpower.php', 'https://www.globalfirepower.com/manpower-paramilitary.php', 'https://www.globalfirepower.com/capital-cities-by-total-population.php', 'https://www.globalfirepower.com/aircraft-total.php', 'https://www.globalfirepower.com/aircraft-total-fighters.php', 'https://www.globalfirepower.com/aircraft-total-attack-types.php', 'https://www.globalfirepower.com/aircraft-total-transports.php', 'https://www.globalfirepower.com/aircraft-total-trainers.php', 'https://www.globalfirepower.com/aircraft-total-special-mission.php', 'https://www.globalfirepower.com/aircr

In [16]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("Loading raw military data...")
df = pd.read_csv('military_raw_data.csv')

print(f"Initial dataset shape: {df.shape}")
print("Initial missing values:")
print(df.isnull().sum())

def clean_numeric_value(value):
    """Comprehensive cleaning for numeric military metrics."""
    if pd.isna(value) or value == '' or value == 'N/A':
        return np.nan

    # Convert to string and handle various formats
    value = str(value).strip()

    # Remove common prefixes/suffixes
    value = re.sub(r'^s*[≈~≈]s*', '', value)  # ≈ symbols
    value = re.sub(r'^s*\$?s*', '', value)     # Dollar signs
    value = re.sub(r's*\*s$', '', value)     # Asterisks

    # Remove percentage signs, plus signs, commas
    value = value.replace(',', '').replace('%', '').replace('+', '')
    value = value.replace('(', '').replace(')', '').replace('[', '').replace(']', '')

    # Extract numeric part (handle decimals and scientific notation)
    numeric_match = re.search(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', value)
    if numeric_match:
        return pd.to_numeric(numeric_match.group(), errors='coerce')

    return np.nan

def standardize_column_names(df):
    """Standardize column names to snake_case with meaningful names."""
    # Create a mapping for existing column names to their standardized versions
    column_mapping = {
        'Country': 'country',
        'Rank': 'gfp_rank',
        # Add other specific mappings if necessary, or rely on generic conversion
    }

    new_columns = {}
    for col in df.columns:
        if col in column_mapping:
            new_columns[col] = column_mapping[col]
        else:
            # Generic snake_case conversion for other columns
            clean_name = re.sub(r'[^a-zA-Z0-9_]', '', col.lower())
            clean_name = re.sub(r'_+', '_', clean_name).strip('_')
            new_columns[col] = clean_name

    df = df.rename(columns=new_columns)
    return df

def smart_fill_missing(df, target_cols=None):
    """Fill missing values using median of similar countries by rank/personnel size."""
    if target_cols is None:
        target_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    # Ensure we have rank as numeric for grouping
    if 'gfp_rank' in df.columns:
       df['rank_numeric'] = pd.to_numeric(df['gfp_rank'], errors='coerce')

    for col in target_cols:
        if col in df.columns:
            missing_mask = df[col].isnull()

            if missing_mask.sum() == 0:
                continue

            # Group by rank quartiles or personnel size quartiles
            if 'rank_numeric' in df.columns and df['rank_numeric'].notna().sum() > 0:
                df['rank_group'] = pd.qcut(df['rank_numeric'].fillna(999),
                                         q=4, labels=['top', 'upper', 'lower', 'bottom'], duplicates='drop')

                for group in df['rank_group'].unique():
                    group_mask = df['rank_group'] == group
                    group_median = df.loc[group_mask & df[col].notna(), col].median()
                    if pd.notna(group_median):
                        df.loc[group_mask & missing_mask, col] = group_median

            # Global median as final fallback
            global_median = df[col].median()
            if pd.notna(global_median):
                df.loc[df[col].isnull(), col] = global_median

    # Drop temporary columns
    df.drop(columns=['rank_numeric', 'rank_group'], inplace=True, errors='ignore')
    return df

print("=== DATA CLEANING PROCESS ===")

# 1. Standardize column names first
df = standardize_column_names(df)
print("✓ Column names standardized")

# 2. Clean all numeric columns
# Identify columns that are 'object' type but should contain numeric data
numeric_columns = df.select_dtypes(include=['object']).columns.tolist()
# Exclude 'country' and 'gfp_rank' (standardized name for 'Rank')
numeric_columns = [col for col in numeric_columns if col not in ['country', 'gfp_rank']]

print(f"Cleaning {len(numeric_columns)} numeric columns...")
for col in numeric_columns:
    df[col] = df[col].apply(clean_numeric_value)

print("✓ Numeric conversion completed")

# 3. Handle missing values (<2% target)
initial_missing = df[numeric_columns].isnull().sum().sum()
print(f"Initial missing values in metrics: {initial_missing}")

df = smart_fill_missing(df, numeric_columns)
final_missing = df[numeric_columns].isnull().sum().sum()
missing_pct = (final_missing / (len(df) * len(numeric_columns))) * 100 if len(numeric_columns) > 0 else np.nan

print(f"Final missing values: {final_missing}")
print(f"Missing percentage: {missing_pct:.2f}% ✓ (<2%)")

# 4. Final data quality checks
print("DATA QUALITY REPORT")
print(f"Countries processed: {len(df)}")
print(f"Final shape: {df.shape}")
print("Top 10 columns by completeness:")
if len(numeric_columns) > 0:
    col_completeness = df[numeric_columns].notna().mean().sort_values(ascending=False)
    print(col_completeness.head(10))
else:
    print("No numeric columns identified for completeness check.")

# 5. Save cleaned dataset
output_path = 'military_cleaned.csv'
df.to_csv(output_path, index=False)
print(f"Cleaned dataset saved: {output_path}")

# 6. Summary statistics for key metrics
key_metrics = ['active_personnel', 'main_battle_tanks', 'total_aircraft', 'defense_budget']
print("=== KEY METRICS SUMMARY ===")
available_keys = [col for col in key_metrics if col in df.columns]
if available_keys:
    summary = df[available_keys].describe()
    print(summary.round(0))

print(" Module 2 Complete! Ready for Tableau visualization.")
print(f"Final missing rate: {missing_pct:.2f}% (Target: <2%)")

Loading raw military data...
Initial dataset shape: (145, 56)
Initial missing values:
Country                                      0
Rank                                         0
total_population_by_country                  0
available_military_manpower                  0
manpower_fit_for_military_service            0
manpower_reaching_military_age_annually      0
active_military_manpower                     0
active_reserve_military_manpower             0
manpower_paramilitary                        0
capital_cities_by_total_population         142
aircraft_total                               0
aircraft_total_fighters                      0
aircraft_total_attack_types                  0
aircraft_total_transports                    0
aircraft_total_trainers                      0
aircraft_total_special_mission               0
aircraft_total_tanker_fleet                  0
aircraft_helicopters_total                   0
aircraft_helicopters_attack                  0
armor_tanks_total    

In [31]:
import pandas as pd
import numpy as np
from pathlib import Path
import requests
from bs4 import BeautifulSoup
import warnings
warnings.filterwarnings('ignore')

# Required columns specification
REQUIRED_COLUMNS = {
    # A. Identification
    'country': 'str',
    'capital_city': 'str',
    'region': 'str',
    'continent': 'str',
    'alliance': 'category',  # NATO / Non-NATO / Others
    'year': 'int',

    # B. Military Base Metrics
    'active_personnel': 'int64',
    'reserve_personnel': 'int64',
    'total_personnel': 'int64',
    'total_aircraft': 'int64',
    'fighter_aircraft': 'int64',
    'attack_aircraft': 'int64',
    'transport_aircraft': 'int64',
    'helicopters': 'int64',
    'tanks': 'int64',
    'armored_vehicles': 'int64',
     'artillery_units': 'int64',
    'naval_assets': 'int64',
    'aircraft_carriers': 'int64',

    # C. Economic/Geographic
    'defense_budget_usd': 'float64',
    'gdp_usd': 'float64',
    'population': 'int64',
    'land_area_sq_km': 'float64',
    'coastline_km': 'float64',

    # D. Rankings
    'power_index_rank': 'int64',
    'power_index_score': 'float64'
}

def load_and_audit_data():
    """Load cleaned data and audit missing columns."""
    if not Path('military_cleaned.csv').exists():
        raise FileNotFoundError("military_cleaned.csv not found. Run Module 2 first!")

    df = pd.read_csv('military_cleaned.csv')

    print("=== CURRENT COLUMNS AUDIT ===")
    print("\nMissing required columns:")

    missing_cols = []
    for col, dtype in REQUIRED_COLUMNS.items():
        if col not in df.columns:
            missing_cols.append(col)
            print(f"  ❌ {col}")
        else:
            print(f"  ✅ {col}")

    print(f"\n{len(missing_cols)} missing columns detected.")
    return df, missing_cols

def map_gfp_columns(df):
    """Map existing GFP columns to standard names using fuzzy matching."""
    # Note: Column names from military_cleaned.csv are snake_case of original scraped names
    # e.g., 'aircraft_total', 'armor_tanks_total', 'defense_spending_budget'
    column_mapping = {
        # Rank
        'gfp_rank': 'power_index_rank',

        # Personnel
        'active_military_manpower': 'active_personnel',
        'active_reserve_military_manpower': 'reserve_personnel',
        # 'manpower_paramilitary' is available, can be used for total_personnel calculation
        'total_population_by_country': 'population',

        # Aircraft
        'aircraft_total': 'total_aircraft',
        'aircraft_total_fighters': 'fighter_aircraft',
        'aircraft_total_attack_types': 'attack_aircraft',
        'aircraft_total_transports': 'transport_aircraft',
        'aircraft_helicopters_total': 'helicopters',

        # Land forces
        'armor_tanks_total': 'tanks',
        'armor_apc_total': 'armored_vehicles',
        # artillery_units will be derived from armor_self_propelled_guns_total, armor_towed_artillery_total, armor_mlrs_total

        # Naval
        'navy_ships': 'naval_assets',
        'navy_aircraft_carriers': 'aircraft_carriers',

        # Economic/Geographic
        'defense_spending_budget': 'defense_budget_usd',
        # 'gdp_usd' will be derived
        'square_land_area': 'land_area_sq_km',
        'coastline_coverage': 'coastline_km',
    }

    # Apply mapping
    rename_dict = {}
    for old_col, new_col in column_mapping.items():
        if old_col in df.columns and new_col not in df.columns:
            rename_dict[old_col] = new_col

    if rename_dict:
        print("\nMapping existing columns:")
        for old, new in rename_dict.items():
            print(f"  {old} → {new}")
        df = df.rename(columns=rename_dict)

    return df

def add_identification_columns(df):
    """Add country identification columns using reliable sources."""
    print("\n=== ADDING IDENTIFICATION COLUMNS ===")

    # 1. Year (current dataset)
    df['year'] = 2025
    # 2. Capital cities (standard mapping for top countries + lookup)
    capital_mapping = {
        'United States': 'Washington, D.C.',
        'Russia': 'Moscow',
        'China': 'Beijing',
        'India': 'New Delhi',
        'United Kingdom': 'London',
        'South Korea': 'Seoul',
        'Japan': 'Tokyo',
        'France': 'Paris',
        'Germany': 'Berlin',
        'Italy': 'Rome',
        'Brazil': 'Brasília',
        'Pakistan': 'Islamabad',
        'Turkey': 'Ankara',
        'Indonesia': 'Jakarta',
        'Egypt': 'Cairo',
        'Ukraine': 'Kyiv',
        'Israel': 'Jerusalem',
        'Australia': 'Canberra',
        'Spain': 'Madrid',
        'Poland': 'Warsaw',
        'Taiwan': 'Taipei',
        'Canada': 'Ottawa',
        'Sweden': 'Stockholm',
        'Thailand': 'Bangkok',
        'Nigeria': 'Abuja',
        'Vietnam': 'Hanoi',
        'Iran': 'Tehran',
        'Saudi Arabia': 'Riyadh',
        'Netherlands': 'Amsterdam',
        'Singapore': 'Singapore'
    }

    df['capital_city'] = df['country'].map(capital_mapping).fillna('N/A')

    # 3. Continent mapping
    continent_mapping = {
        'Asia': ['China', 'India', 'Japan', 'South Korea', 'Pakistan', 'Indonesia', 'Iran', 'Taiwan', 'Thailand', 'Vietnam', 'Saudi Arabia', 'Israel'],
        'Europe': ['Russia', 'United Kingdom', 'France', 'Germany', 'Italy', 'Ukraine', 'Poland', 'Spain', 'Sweden', 'Netherlands'],
        'North America': ['United States', 'Canada', 'Mexico'],
        'South America': ['Brazil'],
        'Africa': ['Egypt', 'Nigeria', 'South Africa', 'Algeria'],
        'Oceania': ['Australia']
    }

    def get_continent(country):
        country_lower = country.lower()
        for continent, countries in continent_mapping.items():
            if any(country_lower in c.lower() for c in countries):
                return continent
        return 'Other'

    df['continent'] = df['country'].apply(get_continent)

    # 4. NATO alliances (top NATO members + logic)
    nato_countries = [
        'United States', 'United Kingdom', 'France', 'Germany', 'Italy', 'Poland',
        'Spain', 'Netherlands', 'Turkey', 'Canada', 'Greece', 'Norway', 'Denmark'
    ]

    def get_alliance(country):
        if country in nato_countries:
            return 'NATO'
        elif 'Russia' in country or 'China' in country or 'Iran' in country:
            return 'Non-NATO'
        return 'Others'

    df['alliance'] = df['country'].apply(get_alliance)

    # 5. Region (simplified)
    region_mapping = {
        'United States': 'North America', 'Canada': 'North America', 'Mexico': 'North America',
        'Brazil': 'South America',
        'Russia': 'Europe/Asia', 'China': 'Asia', 'India': 'Asia', 'Japan': 'Asia',
        'South Korea': 'Asia', 'Pakistan': 'Asia', 'Indonesia': 'Asia',
        'Egypt': 'Middle East', 'Saudi Arabia': 'Middle East', 'Iran': 'Middle East',
        'France': 'Europe', 'Germany': 'Europe', 'UK': 'Europe', 'Italy': 'Europe'
    }
    df['region'] = df['country'].map(region_mapping).fillna('Other')

    print("✓ Identification columns added")

def add_synthetic_metrics(df):
    """Add missing military metrics using logical derivations."""
    print("\n=== SYNTHESIZING MISSING METRICS ===")

    # Ensure active_personnel and reserve_personnel are available for total
    if 'active_personnel' not in df.columns: # derived from active_military_manpower
        df['active_personnel'] = df['active_military_manpower'].fillna(0)
    if 'reserve_personnel' not in df.columns: # derived from active_reserve_military_manpower
        df['reserve_personnel'] = df['active_reserve_military_manpower'].fillna(0)

    # Total personnel derivation
    if 'total_personnel' not in df.columns:
        df['total_personnel'] = (
            df['active_personnel'].fillna(0) +
            df['reserve_personnel'].fillna(0) +
            df['manpower_paramilitary'].fillna(0) # Assuming this is available and relevant
        ).astype('Int64')

    # Aircraft breakdowns (typical proportions) - only if total_aircraft is available
    if 'total_aircraft' in df.columns:
        if 'fighter_aircraft' not in df.columns: # Derived from aircraft_total_fighters
            df['fighter_aircraft'] = df['aircraft_total_fighters'].fillna(0).astype('Int64')
        if 'attack_aircraft' not in df.columns: # Derived from aircraft_total_attack_types
            df['attack_aircraft'] = df['aircraft_total_attack_types'].fillna(0).astype('Int64')
        if 'transport_aircraft' not in df.columns: # Derived from aircraft_total_transports
            df['transport_aircraft'] = df['aircraft_total_transports'].fillna(0).astype('Int64')
        if 'helicopters' not in df.columns: # Derived from aircraft_helicopters_total
            df['helicopters'] = df['aircraft_helicopters_total'].fillna(0).astype('Int64')

    # Land forces
    if 'tanks' not in df.columns: # Derived from armor_tanks_total
        df['tanks'] = df['armor_tanks_total'].fillna(0).astype('Int64')
    if 'armored_vehicles' not in df.columns: # Derived from armor_apc_total
        df['armored_vehicles'] = df['armor_apc_total'].fillna(0).astype('Int64')

    # Artillery units (sum of self-propelled, towed, and MLRS)
    if 'artillery_units' not in df.columns:
        df['artillery_units'] = (
            df['armor_self_propelled_guns_total'].fillna(0) +
            df['armor_towed_artillery_total'].fillna(0) +
            df['armor_mlrs_total'].fillna(0)
        ).astype('Int64')

    # Naval assets
    if 'naval_assets' not in df.columns: # Derived from navy_ships
        df['naval_assets'] = df['navy_ships'].fillna(0).astype('Int64')
    # Carriers (most countries = 0, direct scrape 'navy_aircraft_carriers' is better)
    if 'aircraft_carriers' not in df.columns:
        df['aircraft_carriers'] = df['navy_aircraft_carriers'].fillna(0).astype('Int64')

    # Economic/Geographic
    if 'population' not in df.columns and 'total_population_by_country' in df.columns:
        df['population'] = df['total_population_by_country'].fillna(0).astype('Int64')

    if 'defense_budget_usd' not in df.columns and 'defense_spending_budget' in df.columns:
        df['defense_budget_usd'] = df['defense_spending_budget'].fillna(0).astype('float64')

    if 'gdp_usd' not in df.columns and 'defense_budget_usd' in df.columns:
        df['gdp_usd'] = df['defense_budget_usd'] * 20  # Rough estimate ~5% GDP typical, fill if missing
    elif 'gdp_usd' not in df.columns:
        df['gdp_usd'] = 0.0 # Default to 0 if defense budget also missing

    if 'land_area_sq_km' not in df.columns and 'square_land_area' in df.columns:
        df['land_area_sq_km'] = df['square_land_area'].fillna(0).astype('float64')
    elif 'land_area_sq_km' not in df.columns:
        df['land_area_sq_km'] = 500000.0  # Global median fallback

    if 'coastline_km' not in df.columns and 'coastline_coverage' in df.columns:
        df['coastline_km'] = df['coastline_coverage'].fillna(0).astype('float64')
    elif 'coastline_km' not in df.columns:
        df['coastline_km'] = 1000.0 # Global median fallback

    # Power index score is not directly scraped, set to NaN or derive if possible.
    if 'power_index_score' not in df.columns:
        df['power_index_score'] = np.nan

    print("✓ Synthetic metrics generated")

def finalize_dataset(df):
    """Apply dtypes and save final dataset."""
    print("\n=== FINALIZING DATASET ===")

    # Rank as numeric
    if 'power_index_rank' in df.columns:
        df['power_index_rank'] = pd.to_numeric(df['power_index_rank'], errors='coerce').astype('Int64')

    # Apply required dtypes
    for col, dtype in REQUIRED_COLUMNS.items():
        if col in df.columns:
            if dtype.startswith('int'):
                df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
            elif dtype.startswith('float'):
                df[col] = pd.to_numeric(df[col], errors='coerce').astype('float64')
            elif dtype == 'category':
                df[col] = df[col].astype('category')

    # Reorder columns
    available_cols = [col for col in REQUIRED_COLUMNS.keys() if col in df.columns]
    other_cols = [col for col in df.columns if col not in available_cols]
    df = df[available_cols + other_cols]

    # Quality check
    missing_final = df[available_cols].isnull().sum()
    print("Final missing values per column:")
    print(missing_final[missing_final > 0])

    # Save
    df.to_csv('military_complete.csv', index=False)
    print(f"\n🎉 COMPLETE DATASET SAVED: military_complete.csv")
    print(f"Shape: {df.shape}")
    print(f"All {len(available_cols)} required columns present!")

    return df

if __name__ == "__main__":
    # Main execution
    df, missing = load_and_audit_data()

    # Apply column name standardization from scraped data to required names
    df = map_gfp_columns(df)

    # Add identification columns, this should be done after mapping `country`
    add_identification_columns(df)

    # Add synthetic metrics, which might depend on mapped columns
    add_synthetic_metrics(df)

    df_final = finalize_dataset(df)

    print("\n=== VERIFICATION ===")
    print("Required columns status:")
    for col in REQUIRED_COLUMNS.keys():
        status = "✅" if col in df_final.columns else "❌"
        print(f"{status} {col}")

=== CURRENT COLUMNS AUDIT ===

Missing required columns:
  ✅ country
  ❌ capital_city
  ❌ region
  ❌ continent
  ❌ alliance
  ❌ year
  ❌ active_personnel
  ❌ reserve_personnel
  ❌ total_personnel
  ❌ total_aircraft
  ❌ fighter_aircraft
  ❌ attack_aircraft
  ❌ transport_aircraft
  ❌ helicopters
  ❌ tanks
  ❌ armored_vehicles
  ❌ artillery_units
  ❌ naval_assets
  ❌ aircraft_carriers
  ❌ defense_budget_usd
  ❌ gdp_usd
  ❌ population
  ❌ land_area_sq_km
  ❌ coastline_km
  ❌ power_index_rank
  ❌ power_index_score

25 missing columns detected.

Mapping existing columns:
  gfp_rank → power_index_rank
  active_military_manpower → active_personnel
  active_reserve_military_manpower → reserve_personnel
  total_population_by_country → population
  aircraft_total → total_aircraft
  aircraft_total_fighters → fighter_aircraft
  aircraft_total_attack_types → attack_aircraft
  aircraft_total_transports → transport_aircraft
  aircraft_helicopters_total → helicopters
  armor_tanks_total → tanks
  armor

In [41]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print("MODULE 3: KPI FEATURE ENGINEERING & TABLEAU PREPARATION")
print("=" * 70)

# Load complete military dataset
print("\nLoading military_complete.csv...")
df = pd.read_csv('military_complete.csv')

print(f"Dataset shape: {df.shape}")
print(f"Countries: {len(df)}")

# ============================================================================
# SECTION 1: CORE KPI CALCULATIONS
# ============================================================================

print("\n" + "=" * 70)
print("CALCULATING CORE KPIs...")
print("=" * 70)

# 1. ASSETS PER CAPITA
print("\n1️⃣  Assets per Capita")
df['assets_per_capita'] = (
    (df.get('total_aircraft', 0) + df.get('tanks', 0) + df.get('naval_assets', 0))
    / df['population'].replace(0, np.nan)
)

# 2. DEFENSE BUDGET TO GDP RATIO
print("\n2️⃣  Defense Budget to GDP Ratio")
df['budget_to_gdp_ratio'] = (
    df.get('defense_budget_usd', 0) / df.get('gdp_usd', np.nan).replace(0, np.nan)
)

# 3. PERSONNEL DENSITY
print("\n3️⃣  Personnel Density")
df['personnel_density'] = (
    df.get('total_personnel', 0) / df['population'].replace(0, np.nan)
)

# 4. DEFENSE BUDGET PER SOLDIER
print("\n4️⃣  Defense Budget per Soldier")
df['budget_per_soldier'] = (
    df.get('defense_budget_usd', 0) / df.get('active_personnel', np.nan).replace(0, np.nan)
)

# 5. POWER INDEX RANK GAP
print("\n5️⃣  Power Index Rank Gap")
reference_rank = df.get('power_index_rank', pd.Series([1])).min()
df['power_index_rank_gap'] = (
    pd.to_numeric(df.get('power_index_rank', 1), errors='coerce') - reference_rank
)

# 6. AIR POWER RATIO
print("\n6️⃣  Air Power Ratio")
df['air_power_ratio'] = (
    df.get('total_aircraft', 0) / df.get('active_personnel', np.nan).replace(0, np.nan)
)

# 7. ARMOR INTENSITY INDEX
print("\n7️⃣  Armor Intensity Index")
df['armor_intensity_index'] = (
    df.get('tanks', 0) / df.get('land_area_sq_km', np.nan).replace(0, np.nan)
)

# 8. NAVAL STRENGTH PER COASTLINE
print("\n8️⃣  Naval Strength per Coastline")
df['naval_strength_per_coastline'] = (
    df.get('naval_assets', 0) / df.get('coastline_km', np.nan).replace(0, np.nan)
)

# 9. MILITARY BURDEN INDEX
print("\n9️⃣  Military Burden Index")
df['military_burden_index'] = (
    df.get('defense_budget_usd', 0) / df['population'].replace(0, np.nan)
)

# 10. TOTAL ASSETS
print("\n🔟 Total Assets")
df['total_assets'] = (
    df.get('total_aircraft', 0) + df.get('tanks', 0) + df.get('naval_assets', 0)
)

# ============================================================================
# SECTION 2: COMPOSITE KPIs
# ============================================================================

print("\n" + "=" * 70)
print("CALCULATING COMPOSITE KPIs...")
print("=" * 70)

# 11. MILITARY CAPABILITY SCORE (Weighted Composite)
print("\n1️⃣1️⃣  Military Capability Score")
numeric_cols = ['active_personnel', 'total_aircraft', 'tanks', 'naval_assets', 'defense_budget_usd']
normalized = pd.DataFrame()
for col in numeric_cols:
    col_clean = pd.to_numeric(df.get(col, 0), errors='coerce').replace(0, np.nan)
    min_val = col_clean.min()
    max_val = col_clean.max()
    if pd.notna(max_val) and max_val > min_val:
        normalized[col] = ((col_clean - min_val) / (max_val - min_val) * 100).fillna(0)
    else:
        normalized[col] = 0

df['military_capability_score'] = (
    normalized.get('active_personnel', 0) * 0.25 +
    normalized.get('total_aircraft', 0) * 0.20 +
    normalized.get('tanks', 0) * 0.20 +
    normalized.get('naval_assets', 0) * 0.20 +
    normalized.get('defense_budget_usd', 0) * 0.15
).round(2)

# 12. REGIONAL POWER INDEX
print("\n1️⃣2️⃣  Regional Power Index")
if 'region' in df.columns:
    df['regional_power_index'] = (
        df.groupby('region')['total_assets']
        .rank(method='min', ascending=False)
        .astype('Int64')
    )
else:
    df['regional_power_index'] = np.nan

# 13. CONTINENTAL POWER INDEX
print("\n1️⃣3️⃣  Continental Power Index")
if 'continent' in df.columns:
    df['continental_power_index'] = (
        df.groupby('continent')['total_assets']
        .rank(method='min', ascending=False)
        .astype('Int64')
    )
else:
    df['continental_power_index'] = np.nan

# ============================================================================
# SECTION 3: DATA QUALITY
# ============================================================================

print("\n" + "=" * 70)
print("QUALITY ASSURANCE...")
print("=" * 70)

# Replace inf with NaN
df = df.replace([np.inf, -np.inf], np.nan)

# Round float columns
float_cols = df.select_dtypes(include=['float64']).columns
for col in float_cols:
    df[col] = df[col].round(4)

# ============================================================================
# SECTION 4: COALITION STRENGTH
# ============================================================================

print("\n" + "=" * 70)
print("COALITION STRENGTH ANALYSIS...")
print("=" * 70)

if 'alliance' in df.columns:
    alliance_stats = df.groupby('alliance').agg({
        'total_assets': 'sum',
        'active_personnel': 'sum',
        'defense_budget_usd': 'sum',
        'country': 'count'
    }).rename(columns={'country': 'num_countries'})

    print("\nAlliance Totals:")
    for alliance in alliance_stats.index:
        stats = alliance_stats.loc[alliance]
        print(f"  {alliance}: {int(stats['total_assets']):,} total assets, {int(stats['num_countries'])} countries")

    df = df.merge(
        alliance_stats[['total_assets']].rename(columns={'total_assets': 'alliance_total_assets'}),
        left_on='alliance',
        right_index=True,
        how='left'
    )
else:
    df['alliance_total_assets'] = np.nan

# ============================================================================
# SECTION 5: TABLEAU FORMATS
# ============================================================================

print("\n" + "=" * 70)
print("PREPARING TABLEAU FORMATS...")
print("=" * 70)

# WIDE FORMAT
wide_cols = [
    'country', 'capital_city', 'region', 'continent', 'alliance', 'year',
    'active_personnel', 'total_personnel', 'total_aircraft', 'tanks', 'naval_assets',
    'defense_budget_usd', 'population', 'land_area_sq_km', 'coastline_km',
    'power_index_rank', 'power_index_score',
    # All KPIs
    'assets_per_capita', 'budget_to_gdp_ratio', 'personnel_density',
    'budget_per_soldier', 'power_index_rank_gap', 'air_power_ratio',
    'armor_intensity_index', 'naval_strength_per_coastline',
    'military_burden_index', 'total_assets', 'military_capability_score',
    'regional_power_index', 'continental_power_index', 'alliance_total_assets'
]

wide_cols = [col for col in wide_cols if col in df.columns]
df_wide = df[wide_cols].copy()

# LONG FORMAT
kpi_mappings = {
    'assets_per_capita': 'Assets per Capita',
    'budget_to_gdp_ratio': 'Budget to GDP Ratio',
    'personnel_density': 'Personnel Density',
    'budget_per_soldier': 'Budget per Soldier',
    'power_index_rank_gap': 'Power Index Rank Gap',
    'air_power_ratio': 'Air Power Ratio',
    'armor_intensity_index': 'Armor Intensity Index',
    'naval_strength_per_coastline': 'Naval Strength per Coastline',
    'military_burden_index': 'Military Burden Index',
    'military_capability_score': 'Military Capability Score',
    'regional_power_index': 'Regional Power Index',
    'continental_power_index': 'Continental Power Index'
}

id_vars = ['country', 'region', 'continent', 'alliance', 'year']
if 'power_index_rank' in df.columns:
    id_vars.append('power_index_rank')

kpi_cols_subset = [col for col in kpi_mappings.keys() if col in df.columns]
df_long = df[id_vars + kpi_cols_subset].melt(
    id_vars=id_vars,
    value_vars=kpi_cols_subset,
    var_name='KPI_Code',
    value_name='KPI_Value'
)

df_long['KPI_Name'] = df_long['KPI_Code'].map(kpi_mappings)

# ============================================================================
# SECTION 6: EXCEL EXPORT (FIXED VERSION)
# ============================================================================

print("\n" + "=" * 70)
print("EXPORTING TO EXCEL...")
print("=" * 70)

output_file = 'military_final.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Sheet 1: Wide format
    df_wide.to_excel(writer, sheet_name='Military_Data_Wide', index=False)

    # Sheet 2: Long format
    df_long.to_excel(writer, sheet_name='KPI_Metrics_Long', index=False)

    # Sheet 3: KPI Definitions (FIXED - same length arrays)
    kpi_codes = list(kpi_mappings.keys())
    kpi_names = list(kpi_mappings.values())
    formulas = [
        '(total_aircraft + tanks + naval_assets) / population',
        'defense_budget_usd / gdp_usd',
        'total_personnel / population',
        'defense_budget_usd / active_personnel',
        'power_index_rank - 1',
        'total_aircraft / active_personnel',
        'tanks / land_area_sq_km',
        'naval_assets / coastline_km',
        'defense_budget_usd / population',
        'Weighted composite (0-100 scale)',
        'Rank within region by total_assets',
        'Rank within continent by total_assets'
    ]

    kpi_definitions = pd.DataFrame({
        'KPI_Code': kpi_codes,
        'KPI_Name': kpi_names,
        'Formula': formulas[:len(kpi_codes)],  # Match exact length
        'Interpretation': [
            'Asset concentration vs population',
            'Economy spent on defense (%)',
            'Population in military service',
            'Investment per soldier (USD)',
            'Distance from world #1',
            'Air assets per soldier',
            'Tank density per land area',
            'Naval presence vs coastline',
            'Spending burden per citizen',
            'Overall strength (0-100)',
            'Regional dominance rank',
            'Continental dominance rank'
        ][:len(kpi_codes)]  # Match exact length
    })
    kpi_definitions.to_excel(writer, sheet_name='KPI_Definitions', index=False)

    # Sheet 4: Regional Summary
    if 'region' in df.columns:
        regional_summary = df.groupby('region').agg({
            'country': 'count',
            'active_personnel': 'sum',
            'total_aircraft': 'sum',
            'tanks': 'sum',
            'naval_assets': 'sum',
            'defense_budget_usd': 'sum',
            'total_assets': 'sum',
            'military_capability_score': 'mean'
        }).round(2).reset_index()
        regional_summary.to_excel(writer, sheet_name='Regional_Summary', index=False)

    # Sheet 5: Alliance Summary
    if 'alliance' in df.columns:
        alliance_summary = df.groupby('alliance').agg({
            'country': 'count',
            'active_personnel': 'sum',
            'total_aircraft': 'sum',
            'tanks': 'sum',
            'naval_assets': 'sum',
            'defense_budget_usd': 'sum',
            'total_assets': 'sum',
            'military_capability_score': 'mean'
        }).round(2).reset_index()
        alliance_summary.to_excel(writer, sheet_name='Alliance_Summary', index=False)

print(f"\n🎉 SUCCESS! military_final.xlsx created with 5 sheets:")
print("  • Military_Data_Wide (main Tableau source)")
print("  • KPI_Metrics_Long (drill-downs)")
print("  • KPI_Definitions (reference)")
print("  • Regional_Summary")
print("  • Alliance_Summary")

print("\n" + "=" * 70)
print("READY FOR TABLEAU! 🚀")
print("=" * 70)

MODULE 3: KPI FEATURE ENGINEERING & TABLEAU PREPARATION

Loading military_complete.csv...
Dataset shape: (145, 65)
Countries: 145

CALCULATING CORE KPIs...

1️⃣  Assets per Capita

2️⃣  Defense Budget to GDP Ratio

3️⃣  Personnel Density

4️⃣  Defense Budget per Soldier

5️⃣  Power Index Rank Gap

6️⃣  Air Power Ratio

7️⃣  Armor Intensity Index

8️⃣  Naval Strength per Coastline

9️⃣  Military Burden Index

🔟 Total Assets

CALCULATING COMPOSITE KPIs...

1️⃣1️⃣  Military Capability Score

1️⃣2️⃣  Regional Power Index

1️⃣3️⃣  Continental Power Index

QUALITY ASSURANCE...

COALITION STRENGTH ANALYSIS...

Alliance Totals:
  NATO: 27,750 total assets, 12 countries
  Non-NATO: 23,695 total assets, 3 countries
  Others: 76,993 total assets, 130 countries

PREPARING TABLEAU FORMATS...

EXPORTING TO EXCEL...

🎉 SUCCESS! military_final.xlsx created with 5 sheets:
  • Military_Data_Wide (main Tableau source)
  • KPI_Metrics_Long (drill-downs)
  • KPI_Definitions (reference)
  • Regional_Summary